**News Sentiment as a Trading Signal: Measuring Predictive Decay Across Holding Horizons**

Syed Sirajuddin · Master of Science in Applied Artificial Intelligence · Shiley Marcos School of Engineering, University of San Diego · AAI-590 Capstone

# Notebook 4 of 5 — Pipeline Design: Aggregation, Signals, and Multi-Horizon Backtesting

This notebook documents the design and construction of the trading pipeline — the Model/Pipeline Design and Building element of the code base and the heart of the report's Methodology section. It covers four stages: (1) aggregating article-level sentiment to a daily per-ticker panel, (2) converting that panel into discrete entry signals, (3) evaluating those signals with a fixed-horizon, point-in-time backtest engine across eight holding horizons, and (4) validating the entire apparatus on synthetic data with a planted effect *before* trusting it on live data.


In [ ]:
# Environment setup: resolve the repository root so `src` imports work
# whether this notebook is run from notebooks/ or the project root.
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
RANDOM_SEED = 42

## 1. Daily Sentiment Aggregation

Article-level scores from Notebook 03 are collapsed to one row per ticker per trading day. Beyond a simple mean, we compute a **confidence-weighted mean**, weighting each article by 1 − P(neutral), so that a day with three decisive articles reads as more strongly toned than a day with three ambiguous ones. Rolling versions over a short window (default three trading days) smooth single-article noise and capture multi-day news narratives; the rolling features use only current and past effective dates, preserving the point-in-time guarantee established at ingestion.


In [ ]:
from src.config import PROCESSED_DIR, UNIVERSE
from src.features.aggregate import daily_sentiment_panel
from src.features.technicals import add_technicals

prices = pd.read_parquet(PROCESSED_DIR / "prices_clean.parquet")
scored = pd.read_parquet(PROCESSED_DIR / "news_scored.parquet")
trading_days = pd.DatetimeIndex(prices["date"].drop_duplicates().sort_values())

panel = daily_sentiment_panel(scored, trading_days, UNIVERSE.tickers)
prices_feat = add_technicals(prices)
panel.tail()

## 2. Signal Generation

The entry rule is deliberately simple and interpretable, because the experimental variable in this project is the **holding horizon**, not signal sophistication — a complex signal would confound the decay measurement with its own idiosyncrasies. A long entry requires three conditions on day *t*, using only information effective on or before *t*: rolling sentiment above the long threshold (+0.35 by default), article support of at least two articles in the window (a tone estimate from a single article is unreliable), and a trend filter requiring the close above its 20-day moving average, which prevents buying positive news in a name that is collapsing. Short entries mirror these conditions and can be disabled to run the strategy long-only.


In [ ]:
from src.signals.generate import generate_signals

signals = generate_signals(panel, prices_feat)
print(f"{len(signals)} signals "
      f"({(signals['direction'] == 1).sum()} long, "
      f"{(signals['direction'] == -1).sum()} short)")
signals.head()

## 3. The Multi-Horizon Backtest Engine

The engine evaluates the *same* signal stream at eight fixed holding horizons — 1, 3, 5, 10, 21, 42, 63, and 126 trading days, i.e., one day to roughly six months — isolating the horizon as the only variable. Its execution model encodes four defenses against the biases that inflate published backtests (Bailey et al., 2014; López de Prado, 2018):

1. **One-bar execution delay.** A signal observed at the close of day *t* is filled at the close of day *t+1*, guaranteeing the information existed before the fill and eliminating same-close look-ahead.
2. **Transaction costs.** Every trade pays a fixed per-side cost (10 basis points by default, 20 round-trip). Costs bind hardest at short horizons, where turnover is highest — precisely where the raw signal is expected to be strongest — so reporting net-of-cost results is essential to the horizon comparison being fair.
3. **No pyramiding, capped capacity.** A ticker cannot be re-entered while a position is open, and simultaneous positions are capped (ten by default) with equal weighting, which keeps the daily portfolio return series realistic rather than assuming unlimited capital.
4. **Chronological out-of-sample split.** Thresholds are tuned only on the pre-2024 sample (Notebook 05); all headline results are quoted on the untouched 2024+ window.

Two result granularities are produced per horizon: a per-trade table (for hit rates and return distributions) and a daily portfolio return series (for Sharpe ratio, computed per Sharpe, 1994, and maximum drawdown).


In [ ]:
from src.config import BACKTEST
from src.backtest.engine import run_all_horizons
from src.backtest.baselines import buy_and_hold, random_signals_null

oos = pd.Timestamp(BACKTEST.oos_start)
signals_oos = signals[signals["date"] >= oos]

results = run_all_horizons(signals_oos, prices)
nulls = {h: random_signals_null(signals_oos, prices, h, n_sims=100)
         for h in BACKTEST.horizons}
bh = buy_and_hold(prices)

### 3.1 Baselines

Two baselines frame every result. **Buy-and-hold** on the equal-weighted universe answers whether the strategy adds anything over passive exposure. The **random-signal null** is the more demanding comparison: for each horizon we simulate 100 strategies with the *same number of trades, the same horizon, and the same cost model*, but with entry dates and tickers drawn at random. This produces a full null distribution of Sharpe ratios, so the sentiment strategy is judged against what "no skill plus identical mechanics" achieves — a direct, simulation-based guard against mistaking cost structure or market drift for signal (Bailey et al., 2014).


## 4. Validating the Apparatus on Synthetic Data

Before any live-data conclusion can be trusted, the measurement apparatus itself must be shown to work. We construct a synthetic market in which prices follow geometric random walks *except* for a planted effect: news-sentiment impulses nudge the following ~5 trading days of returns, with the effect decaying geometrically. Because the ground truth is known by construction, the full pipeline — aggregation, signals, backtests, null comparison — should recover a decay curve that peaks at short horizons and falls into the null band beyond roughly two weeks. It does; the demonstration is reproducible via `scripts/demo_synthetic.py` and is re-run below. If the live-data decay curve later looks different, we can attribute the difference to the data rather than to a defect in the machinery.


In [ ]:
# Reproducible validation of the full engine on a planted, decaying effect.
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "scripts" / "demo_synthetic.py")],
               check=True)

## 5. Event Study

As a complement to the discrete horizon grid, an event study traces the mean cumulative signed return, day by day, for 126 trading days following every signal. This renders the *shape* of the decay directly — an early rise that flattens (or reverses) is the visual signature of short-lived information — and will serve as a headline figure in the Results section.

**Backtest outputs and figures — to be completed after the live-data run.**


In [ ]:
from src.analysis.horizon_decay import (horizon_table, event_study,
                                        plot_decay_curve, plot_event_study,
                                        plot_equity_curves)

table = horizon_table(results, nulls)
display(table.round(3))
plot_decay_curve(table)
plot_event_study(event_study(signals_oos, prices, max_days=126))
plot_equity_curves(results, bh)

---
### Acknowledgment of AI Tool Use

Portions of the code scaffolding and prose in this notebook were drafted with the assistance of Anthropic's Claude (Anthropic, 2026) and subsequently reviewed, tested, and revised by the author, who takes full responsibility for the final content, design decisions, and results. This acknowledgment is provided in accordance with University of San Diego academic integrity guidelines on the use of generative AI tools.

Anthropic. (2026). *Claude* [Large language model]. https://claude.ai

### References

Bailey, D. H., Borwein, J. M., López de Prado, M., & Zhu, Q. J. (2014). Pseudo-mathematics and financial charlatanism: The effects of backtest overfitting on out-of-sample performance. *Notices of the American Mathematical Society, 61*(5), 458–471.

López de Prado, M. (2018). *Advances in financial machine learning.* Wiley.

Sharpe, W. F. (1994). The Sharpe ratio. *The Journal of Portfolio Management, 21*(1), 49–58.
